# XGBoost Hyperparameter Tuning cho CIC-UNSW-NB15
Notebook này thực hiện:
1. Load dữ liệu đã tiền xử lý
2. Huấn luyện baseline XGBoost
3. Random search hyperparameter (24/1944 tổ hợp)
4. Đánh giá model tốt nhất trên test set
5. Lưu model + kết quả

## (Tuỳ chọn) Mount Google Drive nếu data nằm trên Drive
Bỏ comment nếu cần.

In [1]:
# @title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set paths
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/anomaly-data'
OUT_DIR = '/content/drive/MyDrive/Colab Notebooks/anomaly-models'
import os
os.makedirs(OUT_DIR, exist_ok=True)

Mounted at /content/drive


In [2]:
import numpy as np, time, os, json, warnings, functools, random
warnings.filterwarnings('ignore')
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import ParameterGrid
import xgboost as xgb
import torch

print(f'XGBoost version: {xgb.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
USE_GPU = (device.type == 'cuda')
if not USE_GPU:
    print('CẢNH BÁO: Không có GPU. Vào Runtime > Change runtime type > T4 GPU rồi chạy lại từ đầu.')

gpu_params = {'device': 'cuda', 'tree_method': 'hist'} if USE_GPU else {'n_jobs': -1}

CHECKPOINT_PATH = f'{OUT_DIR}/search_checkpoint.json'

n_classes = 10
class_names = ['Benign', 'Analysis', 'Backdoor', 'DoS', 'Exploits',
               'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']

XGBoost version: 3.2.0
Using device: cuda


## Cell 3 — Load dữ liệu

In [3]:
X_train = np.load(f'{DATA_DIR}/X_train.npy').astype(np.float32)
X_val   = np.load(f'{DATA_DIR}/X_val.npy').astype(np.float32)
X_test  = np.load(f'{DATA_DIR}/X_test.npy').astype(np.float32)
y_train = np.load(f'{DATA_DIR}/y_train.npy').flatten().astype(np.int64)
y_val   = np.load(f'{DATA_DIR}/y_val.npy').flatten().astype(np.int64)
y_test  = np.load(f'{DATA_DIR}/y_test.npy').flatten().astype(np.int64)

class_counts = np.bincount(y_train.astype(int))
n_train = len(y_train)
sample_weights = n_train / (n_classes * class_counts)
sw_train = sample_weights[y_train].astype(np.float32)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

Train: (313539, 76), Val: (67188, 76), Test: (67188, 76)


## Cell 4 — Hàm hỗ trợ đánh giá

In [4]:
results = []

def evaluate(name, model, X_test_, y_test_):
    preds = model.predict(X_test_)
    if preds.ndim == 2 and preds.shape[1] > 1:
        preds = np.argmax(preds, axis=1)
    macro = f1_score(y_test_, preds, average='macro')
    wgt   = f1_score(y_test_, preds, average='weighted')
    print(f'  {name:45s} Macro={macro:.4f}  Weighted={wgt:.4f}')
    results.append({'model': name, 'macro_f1': macro, 'weighted_f1': wgt})
    return macro, wgt, preds

def full_report(name, model, X_test_, y_test_):
    macro, wgt, preds = evaluate(name, model, X_test_, y_test_)
    print()
    print(classification_report(y_test_, preds, target_names=class_names, digits=4))
    return macro, wgt

## Cell 5 — Huấn luyện baseline XGBoost

In [5]:
print('\n--- Baseline re-fit ---')
model_base = xgb.XGBClassifier(
    objective='multi:softprob', num_class=n_classes,
    n_estimators=500, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric='mlogloss',
    early_stopping_rounds=20, **gpu_params,
)
model_base.fit(X_train, y_train, sample_weight=sw_train,
               eval_set=[(X_val, y_val)], verbose=False)

full_report('XGBoost baseline (re-fit)', model_base, X_test, y_test)


--- Baseline re-fit ---
  XGBoost baseline (re-fit)                     Macro=0.6033  Weighted=0.9312

                precision    recall  f1-score   support

        Benign     0.9996    0.9779    0.9886     53750
      Analysis     0.3028    0.5690    0.3952        58
      Backdoor     0.4699    0.5735    0.5166        68
           DoS     0.3798    0.4836    0.4255       670
      Exploits     0.8627    0.6752    0.7575      4643
       Fuzzers     0.6345    0.8136    0.7130      4442
       Generic     0.7079    0.7741    0.7395       695
Reconnaissance     0.6950    0.7108    0.7028      2510
     Shellcode     0.2251    0.5968    0.3270       315
         Worms     0.3571    0.6757    0.4673        37

      accuracy                         0.9264     67188
     macro avg     0.5634    0.6850    0.6033     67188
  weighted avg     0.9403    0.9264    0.9312     67188



(0.6032896100474374, 0.9311831161949687)

## Cell 6 — Định nghĩa không gian tham số search

In [6]:
param_grid = {
    'n_estimators': [1000],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1],
    'reg_alpha': [0, 0.1, 1.0],
}

random.seed(42)
all_params = list(ParameterGrid(param_grid))
random.shuffle(all_params)
search_params = all_params[:24]
print(f'Testing {len(search_params)} configurations...')

Testing 24 configurations...


## Cell 7 — Random search (chạy lâu nhất, ~20-40 phút)

In [7]:
# Nạp sẵn 11 kết quả đã chạy bằng CPU trước đó (để không phải chạy lại)
_prev_cpu_results = [
    {'index': 0,  'params': {'max_depth': 8,  'learning_rate': 0.1,  'subsample': 1.0, 'colsample_bytree': 0.8, 'min_child_weight': 3, 'gamma': 0,   'reg_alpha': 1.0, 'n_estimators': 1000}, 'val_macro': 0.6199},
    {'index': 1,  'params': {'max_depth': 6,  'learning_rate': 0.01, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 1, 'gamma': 0,   'reg_alpha': 1.0, 'n_estimators': 1000}, 'val_macro': 0.5209},
    {'index': 2,  'params': {'max_depth': 6,  'learning_rate': 0.01, 'subsample': 1.0, 'colsample_bytree': 1.0, 'min_child_weight': 5, 'gamma': 0,   'reg_alpha': 0,   'n_estimators': 1000}, 'val_macro': 0.5166},
    {'index': 3,  'params': {'max_depth': 8,  'learning_rate': 0.01, 'subsample': 1.0, 'colsample_bytree': 0.6, 'min_child_weight': 3, 'gamma': 0,   'reg_alpha': 0.1, 'n_estimators': 1000}, 'val_macro': 0.5446},
    {'index': 4,  'params': {'max_depth': 4,  'learning_rate': 0.1,  'subsample': 1.0, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'gamma': 0.1, 'reg_alpha': 1.0, 'n_estimators': 1000}, 'val_macro': 0.5566},
    {'index': 5,  'params': {'max_depth': 8,  'learning_rate': 0.01, 'subsample': 0.6, 'colsample_bytree': 0.6, 'min_child_weight': 5, 'gamma': 0.1, 'reg_alpha': 0,   'n_estimators': 1000}, 'val_macro': 0.5608},
    {'index': 6,  'params': {'max_depth': 10, 'learning_rate': 0.01, 'subsample': 1.0, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.1, 'reg_alpha': 0,   'n_estimators': 1000}, 'val_macro': 0.5634},
    {'index': 7,  'params': {'max_depth': 8,  'learning_rate': 0.1,  'subsample': 0.6, 'colsample_bytree': 1.0, 'min_child_weight': 1, 'gamma': 0,   'reg_alpha': 1.0, 'n_estimators': 1000}, 'val_macro': 0.6002},
    {'index': 8,  'params': {'max_depth': 10, 'learning_rate': 0.05, 'subsample': 0.6, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.1, 'reg_alpha': 0,   'n_estimators': 1000}, 'val_macro': 0.6182},
    {'index': 9,  'params': {'max_depth': 8,  'learning_rate': 0.1,  'subsample': 0.6, 'colsample_bytree': 1.0, 'min_child_weight': 1, 'gamma': 0,   'reg_alpha': 0.1, 'n_estimators': 1000}, 'val_macro': 0.6102},
    {'index': 10, 'params': {'max_depth': 10, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0,   'reg_alpha': 1.0, 'n_estimators': 1000}, 'val_macro': 0.6167},
]

# Load checkpoint nếu đã có (resume sau disconnect), nếu chưa có thì khởi tạo từ 11 kết quả CPU trên
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        checkpoint = json.load(f)
    done_results = checkpoint['done_results']
    best_macro = checkpoint['best_macro']
    best_params = checkpoint['best_params']
    done_indices = set(checkpoint['done_indices'])
    print(f'Đã tìm thấy checkpoint: {len(done_indices)}/{len(search_params)} config đã chạy xong.')
else:
    done_results = list(_prev_cpu_results)
    done_indices = set(r['index'] for r in _prev_cpu_results)
    best_macro = max(r['val_macro'] for r in _prev_cpu_results)
    best_params = next(r['params'] for r in _prev_cpu_results if r['val_macro'] == best_macro)
    print(f'Khởi tạo từ {len(done_indices)} kết quả CPU đã chạy trước đó.')

print(f'Best val Macro F1 hiện tại: {best_macro:.4f}')
best_xgb = None

for i, params in enumerate(search_params):
    if i in done_indices:
        continue

    t0 = time.time()
    model = xgb.XGBClassifier(
        objective='multi:softprob', num_class=n_classes,
        random_state=42, eval_metric='mlogloss',
        early_stopping_rounds=20, **gpu_params, **params,
    )
    model.fit(X_train, y_train, sample_weight=sw_train,
              eval_set=[(X_val, y_val)], verbose=False)

    preds = model.predict(X_val)
    macro = f1_score(y_val, preds, average='macro')
    elapsed = time.time() - t0

    print(f'  [{i+1:2d}/{len(search_params)}] {elapsed:4.0f}s  depth={params["max_depth"]} '
          f'lr={params["learning_rate"]} sub={params["subsample"]} col={params["colsample_bytree"]} '
          f'mcw={params["min_child_weight"]} gamma={params["gamma"]} '
          f'alpha={params["reg_alpha"]}  → val_macro={macro:.4f}')

    done_results.append({'index': i, 'params': params, 'val_macro': float(macro), 'elapsed_s': elapsed})
    done_indices.add(i)

    if macro > best_macro:
        best_macro = macro
        best_params = params

    with open(CHECKPOINT_PATH, 'w') as f:
        json.dump({
            'done_results': done_results,
            'best_macro': best_macro,
            'best_params': best_params,
            'done_indices': list(done_indices),
        }, f, indent=2)

print(f'\nBest val Macro F1: {best_macro:.4f}')
print(f'Best params: {best_params}')

Khởi tạo từ 11 kết quả CPU đã chạy trước đó.
Best val Macro F1 hiện tại: 0.6199
  [12/24]   53s  depth=8 lr=0.1 sub=0.8 col=0.8 mcw=1 gamma=0 alpha=1.0  → val_macro=0.6130
  [13/24]   38s  depth=10 lr=0.1 sub=0.6 col=0.8 mcw=5 gamma=0.1 alpha=0.1  → val_macro=0.6110
  [14/24]  100s  depth=10 lr=0.01 sub=1.0 col=0.8 mcw=5 gamma=0 alpha=0  → val_macro=0.5671
  [15/24]  103s  depth=10 lr=0.01 sub=0.8 col=0.6 mcw=3 gamma=0 alpha=0  → val_macro=0.5813
  [16/24]   50s  depth=4 lr=0.01 sub=0.8 col=0.8 mcw=3 gamma=0 alpha=0  → val_macro=0.4899
  [17/24]   86s  depth=10 lr=0.05 sub=0.8 col=0.8 mcw=5 gamma=0 alpha=1.0  → val_macro=0.6117
  [18/24]   89s  depth=10 lr=0.05 sub=1.0 col=1.0 mcw=1 gamma=0.1 alpha=1.0  → val_macro=0.6156
  [19/24]   60s  depth=10 lr=0.1 sub=1.0 col=1.0 mcw=1 gamma=0 alpha=1.0  → val_macro=0.6158
  [20/24]   65s  depth=6 lr=0.05 sub=0.8 col=0.8 mcw=5 gamma=0.1 alpha=1.0  → val_macro=0.5842
  [21/24]   84s  depth=8 lr=0.01 sub=0.8 col=1.0 mcw=5 gamma=0.1 alpha=0  → val_

## Cell 8 — Đánh giá model tốt nhất trên test set

In [8]:
print('\n--- Refit model tốt nhất với best_params ---')
best_xgb = xgb.XGBClassifier(
    objective='multi:softprob', num_class=n_classes,
    random_state=42, eval_metric='mlogloss',
    early_stopping_rounds=20, **gpu_params, **best_params,
)
best_xgb.fit(X_train, y_train, sample_weight=sw_train,
             eval_set=[(X_val, y_val)], verbose=False)

print('\n--- Best XGBoost on test ---')
full_report(f'XGBoost tuned (val_macro={best_macro:.4f})', best_xgb, X_test, y_test)


--- Refit model tốt nhất với best_params ---

--- Best XGBoost on test ---
  XGBoost tuned (val_macro=0.6199)              Macro=0.6308  Weighted=0.9331

                precision    recall  f1-score   support

        Benign     0.9996    0.9782    0.9888     53750
      Analysis     0.2967    0.4655    0.3624        58
      Backdoor     0.6667    0.5588    0.6080        68
           DoS     0.4606    0.4448    0.4525       670
      Exploits     0.8286    0.7275    0.7748      4643
       Fuzzers     0.6371    0.8251    0.7190      4442
       Generic     0.7425    0.7885    0.7648       695
Reconnaissance     0.6585    0.7068    0.6818      2510
     Shellcode     0.3333    0.5175    0.4055       315
         Worms     0.5116    0.5946    0.5500        37

      accuracy                         0.9301     67188
     macro avg     0.6135    0.6607    0.6308     67188
  weighted avg     0.9387    0.9301    0.9331     67188



(0.6307596235049695, 0.9331353537176861)

## Cell 9 — Lưu model và kết quả

In [9]:
best_xgb.save_model(f'{OUT_DIR}/xgboost_best.json')

with open(f'{OUT_DIR}/xgboost_best_params.json', 'w') as f:
    json.dump({'val_macro': float(best_macro), 'params': best_params}, f, indent=2)

with open(f'{OUT_DIR}/all_results.json', 'w') as f:
    json.dump({'baseline': results, 'search': done_results}, f, indent=2)

print(f'Saved model to {OUT_DIR}/xgboost_best.json')
print(f'Saved params to {OUT_DIR}/xgboost_best_params.json')
print(f'Saved all results to {OUT_DIR}/all_results.json')

Saved model to /content/drive/MyDrive/Colab Notebooks/anomaly-models/xgboost_best.json
Saved params to /content/drive/MyDrive/Colab Notebooks/anomaly-models/xgboost_best_params.json
Saved all results to /content/drive/MyDrive/Colab Notebooks/anomaly-models/all_results.json


## Cell 10 — (Tuỳ chọn) Download kết quả về máy

In [10]:
# from google.colab import files
# files.download(f'{OUT_DIR}/xgboost_best.json')
# files.download(f'{OUT_DIR}/xgboost_best_params.json')